In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

In [2]:
text_vectorization = keras.layers.TextVectorization(output_mode = 'multi_hot', standardize = 'lower_and_strip_punctuation', split = 'whitespace') 

In [3]:
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
    "Hola! are you laid-back in Mexico"
]

In [4]:
# Index vocabulary of the text
text_vectorization.adapt(dataset)

In [5]:
# Retrieve the vocabulary
vocabulary = text_vectorization.get_vocabulary()

In [6]:
len(vocabulary)

17

In [7]:
print("Vocabulary:")
print(vocabulary)

Vocabulary:
['[UNK]', np.str_('erase'), np.str_('you'), np.str_('write'), np.str_('then'), np.str_('rewrite'), np.str_('poppy'), np.str_('mexico'), np.str_('laidback'), np.str_('in'), np.str_('i'), np.str_('hola'), np.str_('blooms'), np.str_('are'), np.str_('and'), np.str_('again'), np.str_('a')]


In [8]:
print("Vocabulary:")
print(pd.DataFrame(vocabulary))

Vocabulary:
           0
0      [UNK]
1      erase
2        you
3      write
4       then
5    rewrite
6      poppy
7     mexico
8   laidback
9         in
10         i
11      hola
12    blooms
13       are
14       and
15     again
16         a


In [9]:
# Encode and decode an example sentence
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = text_vectorization(test_sentence)
print(encoded_sentence)

tf.Tensor([1 0 0 1 0 1 0 0 0 0 1 0 0 0 1 1 0], shape=(17,), dtype=int64)


In [10]:
"still" in vocabulary

False

In [11]:
text_vectorization("Sloan, HODL, DMD")

<tf.Tensor: shape=(17,), dtype=int64, numpy=array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])>

In [12]:
# Read data from URL

train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url, index_col=0)
val_df = pd.read_csv(val_url, index_col=0)
test_df = pd.read_csv(test_url, index_col=0)

print(f"""
Train samples: {train_df.shape[0]}
Validation samples: {val_df.shape[0]}
Test samples: {test_df.shape[0]}
""")


Train samples: 48991
Validation samples: 16331
Test samples: 21774



In [13]:
train_df.head()

,Lyric,Genre
0,"Oh, girl. I can't get ready (Can't get ready f...",Pop
1,We met on a rainy evening in the summertime. D...,Pop
2,We carried you in our arms. On Independence Da...,Rock
3,I know he loved you. A long time ago. I ain't ...,Pop
4,Paralysis through analysis. Yellow moral uncle...,Rock


In [14]:
train_df.tail()

,Lyric,Genre
48986,"[Hook]. Beamer, Benz, Or Bentley. Beamer, Benz...",Hip Hop
48987,You never listen to me. I know I'm better off ...,Pop
48988,Things have come to a pretty pass. Our romance...,Pop
48989,"Little baby, on my shoulder. I could fall into...",Pop
48990,Music : Rudolf Schenker. Lyrics: Klaus Meine. ...,Rock


In [15]:
# Let's check the proportion of each label in the training set
train_df['Genre'].value_counts() / train_df.shape[0]

Genre
Rock       0.549448
Pop        0.295136
Hip Hop    0.155416
Name: count, dtype: float64

In [16]:
# Let's turn the target into a dummy vector
y_train = pd.get_dummies(train_df['Genre']).to_numpy()
y_val = pd.get_dummies(val_df['Genre']).to_numpy()
y_test = pd.get_dummies(test_df['Genre']).to_numpy()

In [17]:
y_train

array([[False,  True, False],
       [False,  True, False],
       [False, False,  True],
       ...,
       [False,  True, False],
       [False,  True, False],
       [False, False,  True]], shape=(48991, 3))

In [18]:
# First, we set up our Text Vectorization layer using multi-hot encoding

max_tokens = 5000
text_vectorization = keras.layers.TextVectorization(max_tokens=max_tokens, output_mode='multi_hot')

In [19]:
# The vocabulary that will be indexed is given by the text corpus on our train dataset
text_vectorization.adapt(train_df['Lyric'])

In [20]:
text_vectorization.get_vocabulary()[:20]

['[UNK]',
 np.str_('the'),
 np.str_('you'),
 np.str_('i'),
 np.str_('to'),
 np.str_('and'),
 np.str_('a'),
 np.str_('me'),
 np.str_('it'),
 np.str_('my'),
 np.str_('in'),
 np.str_('im'),
 np.str_('on'),
 np.str_('your'),
 np.str_('that'),
 np.str_('of'),
 np.str_('all'),
 np.str_('be'),
 np.str_('is'),
 np.str_('we')]

In [21]:
text_vectorization.get_vocabulary()[-20:]

[np.str_('eden'),
 np.str_('dagger'),
 np.str_('curve'),
 np.str_('cheddar'),
 np.str_('brew'),
 np.str_('appears'),
 np.str_('vacant'),
 np.str_('universal'),
 np.str_('unholy'),
 np.str_('terrified'),
 np.str_('stickin'),
 np.str_('rumble'),
 np.str_('rug'),
 np.str_('pam'),
 np.str_('os'),
 np.str_('ooohh'),
 np.str_('motto'),
 np.str_('marshall'),
 np.str_('loyalty'),
 np.str_('legacy')]

In [22]:
# We vectorize our input
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [23]:
X_train

<tf.Tensor: shape=(48991, 5000), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], shape=(48991, 5000))>

In [25]:
inputs = keras.Input(shape=(max_tokens, ))
x = keras.layers.Dense(8, activation='relu')(inputs)
outputs = keras.layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │        40,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,035 (156.39 KB)

 Trainable params: 40,035 (156.39 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [27]:
# Fit model

model.fit(x=X_train, y=y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 2s 900us/step - accuracy: 0.7288 - loss: 0.6234 - val_accuracy: 0.7513 - val_loss: 0.5734
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 697us/step - accuracy: 0.7680 - loss: 0.5295 - val_accuracy: 0.7502 - val_loss: 0.5769
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 708us/step - accuracy: 0.7838 - loss: 0.4990 - val_accuracy: 0.7475 - val_loss: 0.5866
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 822us/step - accuracy: 0.7946 - loss: 0.4757 - val_accuracy: 0.7452 - val_loss: 0.5990
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8054 - loss: 0.4544 - val_accuracy: 0.7421 - val_loss: 0.6162
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 843us/step - accuracy: 0.8159 - loss: 0.4333 - val_accuracy: 0.7398 - val_loss: 0.6394
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 655us/step - accuracy: 0.8252 - loss: 0.4144 - val_accuracy: 0.7354 - val_loss: 0.6643
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 1s 771us/step - accuracy: 0.8332 - l

In [28]:
model.evaluate(x=X_test, y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - accuracy: 0.7204 - loss: 0.7663


[0.7663169503211975, 0.7204004526138306]

In [29]:
# Text Vectorization layer using bigrams
text_vectorization = keras.layers.TextVectorization(ngrams=2, output_mode='multi_hot')

In [30]:
text_vectorization.adapt(["the cat sat on the mat."])

In [31]:
text_vectorization.get_vocabulary()

['[UNK]',
 np.str_('the'),
 np.str_('the mat'),
 np.str_('the cat'),
 np.str_('sat on'),
 np.str_('sat'),
 np.str_('on the'),
 np.str_('on'),
 np.str_('mat'),
 np.str_('cat sat'),
 np.str_('cat')]

In [32]:
# Text Vectorization layer using bigrams

text_vectorization = keras.layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,  #note that we are increasing this from 5000 to accommodate bigrams
    output_mode="multi_hot",
)

In [33]:
# We run the STIE process on the training corpus
text_vectorization.adapt(train_df['Lyric'])

In [34]:
text_vectorization.get_vocabulary()[:10]

['[UNK]',
 np.str_('the'),
 np.str_('you'),
 np.str_('i'),
 np.str_('to'),
 np.str_('and'),
 np.str_('a'),
 np.str_('me'),
 np.str_('it'),
 np.str_('my')]

In [35]:
text_vectorization.get_vocabulary()[-10:]

[np.str_('8x'),
 np.str_('44'),
 np.str_('you’re a'),
 np.str_('your mom'),
 np.str_('your god'),
 np.str_('you shot'),
 np.str_('you hell'),
 np.str_('you far'),
 np.str_('ya all'),
 np.str_('x6')]

In [36]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [37]:
inputs = keras.Input(shape=(20000, ))
x = keras.layers.Dense(8, activation='relu')(inputs)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │       160,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,035 (625.14 KB)

 Trainable params: 160,035 (625.14 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [39]:
# Fit model
model.fit(x=X_train, y=y_train, validation_data=(X_val, y_val), epochs=3, batch_size=32)

Epoch 1/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6619 - loss: 0.7561 - val_accuracy: 0.7375 - val_loss: 0.6009
Epoch 2/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7246 - loss: 0.6390 - val_accuracy: 0.7499 - val_loss: 0.5767
Epoch 3/3
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7587 - loss: 0.5705 - val_accuracy: 0.7521 - val_loss: 0.5814


In [40]:
model.evaluate(x=X_test, y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7420 - loss: 0.5948


[0.5948198437690735, 0.7420317530632019]

In [41]:
def lyric_predict(phrase):
    vect_data = text_vectorization([phrase])
    predictions = model.predict(vect_data)
    predictions
    print(f"{float(predictions[0,0] * 100):.2f} % Hip-Hop")
    print(f"{float(predictions[0,1] * 100):.2f} % Pop")
    print(f"{float(predictions[0,2] * 100):.2f} % Rock")

In [42]:
phrase = """
You can dance, you can jive, having the time of your life
See that girl, watch that scene, diggin' the dancing queen
Friday night and the lights are low,
Looking out for the place to go,
Where they play the right music, getting in the swing.
You come in to look for a king.
Anybody could be that guy,
Night is young and the music's high.
With a bit of rock music, everything is fine,
You're in the mood for a dance.
And when you get the chance...
Chorus:
You are the dancing queen, young and sweet, only seventeen.
Dancing queen, feel the beat from the tambourine.
You can dance, you can jive, having the time of your life.
See that girl, watch that scene, diggin' the dancing queen.
You're a teaser, you turn 'em on,
Leave them burning and then you're gone.
Looking out for another, anyone will do,
You're in the mood for a dance.
And when you get the chance...
"""

In [43]:
lyric_predict(phrase)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1.23 % Hip-Hop
68.56 % Pop
30.21 % Rock
